# Kaggle — PlantGuard-style BTC/ETH training

> **Decision-support only. Not financial advice.** Markets are noisy and adversarial. A model that fails to beat baselines after fees/slippage must not be used for trading.

Adapted from the Plant Guard image-classification workflow for **causal BTC/ETH time series**. The 16 sections below mirror that polished flow: GPU check -> config -> dataset validation -> label/sample visualization -> two-phase training -> curves -> confusion matrices -> classification report -> prediction demo -> save -> reload -> deployment smoke test. All logic lives in `src/` — the notebook only orchestrates.


## Kaggle setup

In [ ]:
# --- Kaggle setup ---
# REQUIRES the notebook's "Internet" toggle = ON (Settings → Internet) so we can
# git-clone the public repo and pip-install. Select "GPU T4 x2" for dual-T4 →
# the project auto-detects and uses MirroredStrategy.
import os, sys, subprocess

REPO_URL = "https://github.com/mindees/crypto-ai.git"
BASE = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.getcwd()
REPO_DIR = os.path.join(BASE, "crypto-ai")
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
os.chdir(REPO_DIR)
sys.path.insert(0, os.getcwd())   # make `import src...` work in this kernel
# IMPORTANT: install ONLY the missing extras — do NOT reinstall requirements.txt
# here, or you'll overwrite Kaggle's GPU-matched TensorFlow (2.19) with 2.21 and
# break GPU detection. The project runs on the host's Keras-3 TensorFlow.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-notebook.txt"], check=False)
print("cwd:", os.getcwd())
assert os.path.isdir("src"), "repo not cloned correctly — is Internet enabled?"


In [ ]:
# --- pull the latest prebuilt dataset from Kaggle (if configured) ---
# If you publish processed parquet as a Kaggle dataset, attach it to this
# notebook and symlink it into data/. Otherwise the build step below
# regenerates features/labels from raw (requires the raw parquet to be present).
import os
from pathlib import Path
KAGGLE_INPUT = Path("/kaggle/input")
if KAGGLE_INPUT.exists():
    print("Kaggle input datasets:", [p.name for p in KAGGLE_INPUT.iterdir()])
else:
    print("not on Kaggle or no input datasets attached.")


## 1. Imports & GPU check

In [ ]:
# --- GPU / environment check ---
import json
from src.utils.hardware import detect_hardware
report = detect_hardware()
print(json.dumps(report, indent=2, default=str))

if report.get("would_use_mirrored_strategy"):
    print("\n>>> 2+ GPUs detected — training will use tf.distribute.MirroredStrategy().")
elif report.get("gpu_count", 0) == 1:
    print("\n>>> single GPU — standard single-device training.")
else:
    print("\n>>> no GPU — CPU smoke only. Enable the accelerator for real training.")


## 2. Configuration

In [ ]:
# --- single visible config block (override here, no need to edit src/) ---
ASSETS       = ["BTCUSDT", "ETHUSDT"]
TIMEFRAMES   = ["1h", "4h"]           # train_default per configs/config.yaml
MARKET       = "spot"
START_YEAR   = 2022                    # bound ingestion so the run stays Kaggle/Colab-sized; None = full history
SAMPLE       = False                   # True = last 500 bars (≈2-min smoke); False = full window
PHASE1_EPOCHS = 10
PHASE2_EPOCHS = 25
BATCH_SIZE    = 256
RESUME        = True                   # resume from latest checkpoint if present
print({"assets": ASSETS, "timeframes": TIMEFRAMES, "market": MARKET,
       "start_year": START_YEAR, "sample": SAMPLE, "phase1": PHASE1_EPOCHS,
       "phase2": PHASE2_EPOCHS, "batch_size": BATCH_SIZE, "resume": RESUME})


## 3. Resume support

In [ ]:
# --- resume support: detect the latest run dir with checkpoints ---
from pathlib import Path
runs = sorted((Path("artifacts") / "runs").glob("*")) if (Path("artifacts") / "runs").exists() else []
resumable = [r for r in runs if (r / "phase1_best.keras").exists() or (r / "model.keras").exists()]
if RESUME and resumable:
    LATEST_RUN = resumable[-1].name
    print(f"resumable run found: {LATEST_RUN}")
    print("training will continue from its best checkpoint where supported.")
else:
    LATEST_RUN = None
    print("no prior checkpoint — training from scratch.")


## 3b. Ingest free data + build features/labels
Internet required. OHLCV is bounded by START_YEAR; context adapters (sentiment/coingecko/onchain/derivatives) are resilient.

In [ ]:
# --- ingest free data (Internet required) ---
# OHLCV bounded by START_YEAR to keep the run quick; set START_YEAR=None above
# for the deepest verified history. Context adapters are resilient (non-fatal).
import subprocess, sys
bin_cmd = [sys.executable, "-m", "src.ingest.binance_bulk",
           "--symbols", *ASSETS, "--market-types", MARKET, "--timeframes", *TIMEFRAMES]
if START_YEAR:
    bin_cmd += ["--start-year", str(START_YEAR)]
print(" ".join(bin_cmd))
subprocess.run(bin_cmd, check=True)
subprocess.run([sys.executable, "-m", "src.ingest.sentiment"], check=False)
subprocess.run([sys.executable, "-m", "src.ingest.coingecko"], check=False)
subprocess.run([sys.executable, "-m", "src.ingest.onchain", "--btc-only"], check=False)
subprocess.run([sys.executable, "-m", "src.ingest.derivatives", "--symbols", *ASSETS], check=False)


In [ ]:
# --- build causal features + labels ---
import subprocess, sys
subprocess.run([sys.executable, "-m", "src.features.build_matrix",
                "--symbols", *ASSETS, "--timeframes", *TIMEFRAMES,
                "--market", MARKET, "--sample", "true" if SAMPLE else "false"], check=True)
subprocess.run([sys.executable, "-m", "src.labels.labeling",
                "--symbols", *ASSETS, "--timeframes", *TIMEFRAMES,
                "--market", MARKET, "--sample", "true" if SAMPLE else "false"], check=True)


## 4. Dataset validation + build
Builds windows, prints per-combo row counts, feature counts, and the train/val/test split sizes.

In [ ]:
# --- build the sequence-windowed dataset (train-only scaler, purged splits) ---
import subprocess, sys
cmd = [sys.executable, "-m", "src.datasets.build_dataset",
       "--symbols", *ASSETS, "--timeframes", *TIMEFRAMES,
       "--market", MARKET, "--sample", "true" if SAMPLE else "false"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 5. Label & sample-window visualization
The two-phase trainer emits label-distribution context and sample charts; see artifacts after the run. (Heavy chart code lives in `src/models/train_like_plantguard.py`.)

## 6–9. Two-phase training (Phase 1 warmup + Phase 2 fine-tune)
Produces training curves, confusion matrices (direction/regime/cycle), and the classification report.

In [ ]:
# --- PlantGuard-style two-phase training (calls the real src module) ---
# Phase 1: supervised head warmup. Phase 2: fine-tune last encoder blocks at lower LR.
# Produces: training curves, confusion matrices, classification report,
# prediction demo, reload test — all under artifacts/runs/<id>_plantguard/.
import subprocess, sys
cmd = [sys.executable, "-m", "src.models.train_like_plantguard",
       "--symbols", *ASSETS, "--timeframes", TIMEFRAMES[0],
       "--phase1-epochs", str(PHASE1_EPOCHS), "--phase2-epochs", str(PHASE2_EPOCHS),
       "--batch-size", str(BATCH_SIZE), "--sample", "true" if SAMPLE else "false"]
print(" ".join(cmd))
subprocess.run(cmd, check=True)


## 10–13. Curves, confusion matrices, classification report, prediction demo
All saved under `artifacts/runs/<id>_plantguard/`:
`training_curves.png`, `confusion_*.png`, `classification_report.json`, `prediction_demo.json`.

In [ ]:
from pathlib import Path
runs = sorted((Path("artifacts") / "runs").glob("*_plantguard"))
if runs:
    latest = runs[-1]
    print("artifacts in", latest.name)
    for f in sorted(latest.iterdir()):
        print("  ", f.name)


## 14–15. Save final model + reload test
The trainer already saves `model.keras` and runs a reload + sanity prediction. Confirm it printed `reload + sanity prediction: OK`.

## 16. Deployment smoke test

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "src.models.predict", "--latest",
                "--symbols", *ASSETS, "--timeframes", *TIMEFRAMES], check=False)
subprocess.run([sys.executable, "-m", "src.serve.api", "--smoke-test"], check=False)
subprocess.run([sys.executable, "-m", "src.models.registry", "--sync"], check=False)
print("deployment smoke test complete.")


## Evaluate + backtest + register

In [ ]:
# --- honest evaluation + event-driven backtest vs baselines ---
import subprocess, sys
subprocess.run([sys.executable, "-m", "src.models.evaluate", "--latest",
                "--timeframe", TIMEFRAMES[0], "--sample", "true" if SAMPLE else "false"], check=False)
subprocess.run([sys.executable, "-m", "src.backtest.engine", "--latest",
                "--timeframes", *TIMEFRAMES, "--sample", "true" if SAMPLE else "false"], check=False)
print("Eval + backtest reports written under reports/. "
      "Check whether the model BEATS baselines before trusting it.")


In [ ]:
# --- register the trained run as a candidate; dry-run the promotion gates ---
import subprocess, sys
subprocess.run([sys.executable, "-m", "src.models.registry", "--sync"], check=False)
subprocess.run([sys.executable, "-m", "src.models.registry", "--list"], check=False)
subprocess.run([sys.executable, "-m", "src.models.promote", "--latest", "--dry-run"], check=False)


## Save / export

In [ ]:
# --- export artifacts to /kaggle/working (downloadable / dataset-versionable) ---
import shutil
from pathlib import Path
out = Path("/kaggle/working/artifacts") if Path("/kaggle/working").exists() else Path("artifacts_export")
out.mkdir(parents=True, exist_ok=True)
src_runs = Path("artifacts/runs")
if src_runs.exists():
    latest = sorted(src_runs.glob("*"))[-1]
    shutil.copytree(latest, out / latest.name, dirs_exist_ok=True)
    print(f"exported {latest.name} -> {out}")
print("Also copy reports/ for the eval + backtest summaries.")
